In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from google.colab import files
uploaded = files.upload()

import zipfile
with zipfile.ZipFile("WaRP-C-preprocessed.zip", "r") as z:
    z.extractall("/content/WaRP-C-preprocessed")

PREPROCESSED_ROOT = "/content/WaRP-C-preprocessed"
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

light_train_pipeline = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def _make_flat_dataset(root_dir, transform):
    samples, class_to_idx = [], {}
    for superclass in sorted(os.listdir(root_dir)):
        sp = os.path.join(root_dir, superclass)
        if not os.path.isdir(sp): continue
        for subclass in sorted(os.listdir(sp)):
            scp = os.path.join(sp, subclass)
            if not os.path.isdir(scp): continue
            if subclass not in class_to_idx:
                class_to_idx[subclass] = len(class_to_idx)
            for img_name in os.listdir(scp):
                if img_name.lower().endswith(".jpg"):
                    samples.append((os.path.join(scp, img_name), class_to_idx[subclass]))
    dataset = datasets.ImageFolder(root_dir, transform=transform)
    dataset.samples = dataset.imgs = samples
    dataset.targets = [s[1] for s in samples]
    dataset.classes = list(class_to_idx.keys())
    dataset.class_to_idx = class_to_idx
    return dataset

def get_dataloaders(root=PREPROCESSED_ROOT, batch_size=32, num_workers=2, seed=42):
    torch.manual_seed(seed)
    train_ds = _make_flat_dataset(f"{root}/train", transform=light_train_pipeline)
    val_ds   = _make_flat_dataset(f"{root}/val",   transform=eval_pipeline)
    test_ds  = _make_flat_dataset(f"{root}/test",  transform=eval_pipeline)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds

train_loader, val_loader, test_loader, *_ = get_dataloaders()
num_classes = len(train_loader.dataset.classes)

model = models.regnet_y_400mf(weights=models.RegNet_Y_400MF_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
NUM_EPOCHS = 15

def train_one_epoch(model, loader):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f}")

test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="weighted", zero_division=0)

print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

cm = confusion_matrix(test_labels, test_preds)

Saving WaRP-C-preprocessed.zip to WaRP-C-preprocessed.zip
Downloading: "https://download.pytorch.org/models/regnet_y_400mf-c65dace8.pth" to /root/.cache/torch/hub/checkpoints/regnet_y_400mf-c65dace8.pth


100%|██████████| 16.8M/16.8M [00:00<00:00, 24.8MB/s]


Epoch 1/15 | Train loss 1.6209 acc 0.5057 | Val loss 1.2027 acc 0.6249
Epoch 2/15 | Train loss 1.1110 acc 0.6358 | Val loss 1.1712 acc 0.6312
Epoch 3/15 | Train loss 0.8828 acc 0.7070 | Val loss 1.0210 acc 0.6918
Epoch 4/15 | Train loss 0.7786 acc 0.7445 | Val loss 1.0211 acc 0.6895
Epoch 5/15 | Train loss 0.6554 acc 0.7834 | Val loss 1.0924 acc 0.6646
Epoch 6/15 | Train loss 0.6013 acc 0.8018 | Val loss 1.1796 acc 0.6646
Epoch 7/15 | Train loss 0.5128 acc 0.8260 | Val loss 1.2550 acc 0.6652
Epoch 8/15 | Train loss 0.4394 acc 0.8500 | Val loss 1.2191 acc 0.6816
Epoch 9/15 | Train loss 0.4208 acc 0.8591 | Val loss 1.0847 acc 0.6969
Epoch 10/15 | Train loss 0.3773 acc 0.8706 | Val loss 1.1729 acc 0.6873
Epoch 11/15 | Train loss 0.3240 acc 0.8864 | Val loss 1.2932 acc 0.6924
Epoch 12/15 | Train loss 0.2716 acc 0.9108 | Val loss 1.2550 acc 0.6805
Epoch 13/15 | Train loss 0.3009 acc 0.8999 | Val loss 1.2860 acc 0.6691
Epoch 14/15 | Train loss 0.2333 acc 0.9233 | Val loss 1.3525 acc 0.6708
E